In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json
from numpy.linalg import norm

RUN_DIR = Path("experiments/RN50_20250623_214602")  
IMG_EMB = np.load(RUN_DIR / "img_embs.npy").astype("float32")
TXT_EMB = np.load(RUN_DIR / "txt_embs.npy").astype("float32")
IDS = json.loads((RUN_DIR / "ids.json").read_text())
PARQUET = Path("C:/Users/steph/OneDrive/Desktop/data/metadata.parquet")
META = pd.read_parquet(PARQUET).set_index("id").loc[IDS]

# Normalize embeddings
img_norm = IMG_EMB / norm(IMG_EMB, axis=1, keepdims=True)
txt_norm = TXT_EMB / norm(TXT_EMB, axis=1, keepdims=True)

print("Shapes:", img_norm.shape, txt_norm.shape) 


Shapes: (2000, 1024) (2000, 1024)


In [4]:
# Compute image-to-text recall
def image_to_text_recall(k=1, chunk=2000):
    n = img_norm.shape[0]
    hits = np.zeros(n, dtype=bool)
    for s in range(0, n, chunk):
        e = min(s + chunk, n)
        sim = img_norm[s:e] @ txt_norm.T
        topk = np.argpartition(-sim, k - 1, axis=1)[:, :k]
        rows = np.arange(s, e)[:, None]
        hits[s:e] = np.any(topk == rows, axis=1)
    return hits.mean() * 100


In [5]:
# Generate Recall@k for k=1,5,10
metrics_it = {f"R@{k}": round(image_to_text_recall(k), 2) for k in (1, 5, 10)}
print(metrics_it)

{'R@1': 53.25, 'R@5': 78.4, 'R@10': 88.05}


In [ ]:
# Save metrics to JSON
out_file = RUN_DIR / "image_text_metrics.json"
json.dump(metrics_it, open(out_file, "w"), indent=2)
print("Saved:", out_file, metrics_it)

Saved: experiments\RN50_20250623_214602\image_text_metrics.json {'R@1': 53.25, 'R@5': 78.4, 'R@10': 88.05}
